# Pydantic — Complete Practice Notebook

**Pydantic** is a Python library for **data validation and settings management** using Python type annotations.  
It enforces type hints at runtime, automatically casts compatible types, and provides clear error messages when data is invalid.

---
### Topics Covered
1. Install & Imports
2. Basic Model
3. Automatic Type Casting
4. Default Values
5. Optional Fields
6. Typing — List, Dict, Any, Optional
7. Field() — Constraints & Metadata
8. Nested Models
9. Model Inheritance
10. Serialization — `model_dump`, `model_dump_json`, `model_validate`
11. ConfigDict — Model Configuration
12. Full Examples
13. Practice Cells

---
## Step 1 — Install Pydantic

In [ ]:
!pip install pydantic

---
## Step 2 — Imports

All imports needed throughout this notebook:

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ConfigDict
from typing import List, Dict, Any, Optional, Tuple
import json

print("All imports successful!")

---
## 1. Basic Pydantic Model

Every Pydantic model inherits from `BaseModel`.  
Fields are defined as **class attributes with type annotations** — just like a dataclass, but with built-in validation.

```python
class MyModel(BaseModel):
    field_name: type
```

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str

# Creating an instance
user = User(name="Alice", age=25, email="alice@example.com")

print(user)               # Full model repr
print(user.name)          # Access individual field
print(user.age)
print(type(user.age))     # <class 'int'>

---
## 2. Automatic Type Casting

Pydantic **automatically converts compatible types** when you pass data in.  
For example, passing a `string "25"` where `int` is expected → Pydantic casts it to `25` automatically.

- `"999.99"` → `float` ✅
- `"5"` → `int` ✅
- `"True"` → `bool` ✅
- `"abc"` → `int` ❌ raises `ValidationError`

In [ ]:
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    quantity: int
    in_stock: bool

# All values passed as strings — Pydantic auto-casts them
product = Product(
    name="Laptop",
    price="999.99",      # string → float
    quantity="5",        # string → int
    in_stock="True"      # string → bool
)

print(f"Price: {product.price}  → type: {type(product.price)}")
print(f"Qty:   {product.quantity}  → type: {type(product.quantity)}")
print(f"Stock: {product.in_stock}  → type: {type(product.in_stock)}")

In [ ]:
# What happens when casting fails?
from pydantic import ValidationError

try:
    bad = Product(name="Broken", price="not-a-number", quantity=1, in_stock=True)
except ValidationError as e:
    print("ValidationError caught!")
    print(e)

---
## 3. Default Values

Fields can have **default values**. If the field is not passed in, the default is used automatically.

```python
field_name: type = default_value
```

In [ ]:
from pydantic import BaseModel

class ServerConfig(BaseModel):
    host: str = "localhost"
    port: int = 8080
    debug: bool = False
    max_connections: int = 100

# No values provided — all defaults used
config1 = ServerConfig()
print("Default config:", config1)

# Override some defaults
config2 = ServerConfig(host="0.0.0.0", port=443, debug=True)
print("Custom config: ", config2)

---
## 4. Optional Fields

`Optional[T]` means the field can either hold a value of type `T` **or be `None`**.  
Always pair `Optional` with `= None` as the default, otherwise Pydantic still requires it.

```python
from typing import Optional

field: Optional[str] = None
```

In [ ]:
from pydantic import BaseModel
from typing import Optional

class Employee(BaseModel):
    name: str
    department: str
    manager: Optional[str] = None       # Not required
    salary: Optional[float] = None      # Not required
    linkedin: Optional[str] = None

# Only required fields passed
emp1 = Employee(name="Bob", department="Engineering")
print(emp1)
print(f"Manager: {emp1.manager}")   # None

# Optional fields provided
emp2 = Employee(name="Sara", department="Data", manager="Alice", salary=95000.0)
print(emp2)

---
## 5. Typing — List, Dict, Any, Optional

Pydantic fully supports Python's `typing` module for complex/nested types:

| Type | Meaning |
|---|---|
| `List[str]` | A list where every element must be a string |
| `Dict[str, Any]` | A dict with string keys and values of any type |
| `Any` | Any Python type — no validation on value |
| `Optional[str]` | Either a `str` or `None` |
| `Tuple[int, str]` | A tuple of exactly int then str |

In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Any, Optional

class Report(BaseModel):
    title: str
    tags: List[str]                   # List of strings
    scores: List[float]               # List of floats
    metadata: Dict[str, Any]          # Dict — string keys, any values
    extra_info: Optional[Dict[str, Any]] = None

report = Report(
    title="Q1 Analysis",
    tags=["finance", "quarterly", "2024"],
    scores=[95.5, 88.0, 76.3],
    metadata={"version": 2, "reviewed": True, "author": "Alice"}
)

print("Title   :", report.title)
print("Tags    :", report.tags)
print("Scores  :", report.scores)
print("Metadata:", report.metadata)
print("Extra   :", report.extra_info)  # None — optional, not provided

In [ ]:
# Pydantic also casts inside lists!
report2 = Report(
    title="Test",
    tags=["a", "b"],
    scores=["90", "85.5", "78"],   # strings inside list → cast to float
    metadata={}
)

print(report2.scores)               # [90.0, 85.5, 78.0]
print(type(report2.scores[0]))      # <class 'float'>

---
## 6. Field() — Constraints & Metadata

`Field()` replaces the default value and lets you add **constraints and documentation** to a field.

```python
from pydantic import Field

field: type = Field(default=..., min_length=..., max_length=..., ge=..., le=..., description=...)
```

| Parameter | Works on | Meaning |
|---|---|---|
| `min_length` | `str`, `list` | Minimum length |
| `max_length` | `str`, `list` | Maximum length |
| `gt` | `int`, `float` | Greater than |
| `ge` | `int`, `float` | Greater than or equal to |
| `lt` | `int`, `float` | Less than |
| `le` | `int`, `float` | Less than or equal to |
| `pattern` | `str` | Must match this regex |
| `description` | any | Human-readable description |
| `examples` | any | List of example values |

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class UserProfile(BaseModel):
    username: str = Field(
        min_length=3,
        max_length=20,
        description="Unique username",
        examples=["alice_23", "bob_dev"]
    )
    age: int = Field(
        ge=18,
        le=120,
        description="Age must be between 18 and 120"
    )
    rating: float = Field(
        default=0.0,
        ge=0.0,
        le=5.0,
        description="Rating from 0.0 to 5.0"
    )
    bio: Optional[str] = Field(
        default=None,
        max_length=200,
        description="Short bio"
    )

# Valid user
u = UserProfile(username="shriman_01", age=22, rating=4.8)
print(u)

In [ ]:
from pydantic import ValidationError

# Username too short
try:
    bad_user = UserProfile(username="ab", age=22)
except ValidationError as e:
    print("Error — username too short:")
    print(e)

print("---")

# Age below minimum
try:
    bad_user = UserProfile(username="valid_name", age=15)
except ValidationError as e:
    print("Error — age too low:")
    print(e)

---
## 7. Nested Models

You can use one Pydantic model **as the type of a field** inside another model.  
Pydantic will validate the nested model automatically — you can even pass a plain `dict` and it converts it.

```python
class Inner(BaseModel):
    ...

class Outer(BaseModel):
    inner_field: Inner          # Nested model
    inner_list: List[Inner]     # List of nested models
```

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Address(BaseModel):
    street: str
    city: str
    zip_code: str = Field(min_length=5, max_length=10)
    country: str = "India"

class Order(BaseModel):
    item: str
    quantity: int = Field(ge=1)
    price: float = Field(gt=0)

class Customer(BaseModel):
    name: str
    address: Address           # Nested single model
    orders: List[Order]        # Nested list of models
    vip: bool = False

# Pass dicts — Pydantic converts them automatically
customer = Customer(
    name="Rahul",
    address={"street": "45 MG Road", "city": "Bangalore", "zip_code": "560001"},
    orders=[
        {"item": "Keyboard", "quantity": 1, "price": 1500},
        {"item": "Mouse",    "quantity": 2, "price": 799}
    ]
)

print("Customer :", customer.name)
print("City     :", customer.address.city)       # Access nested field
print("Country  :", customer.address.country)    # Default value
print("1st Item :", customer.orders[0].item)
print("Order qty:", customer.orders[1].quantity)

---
## 8. Model Inheritance

Pydantic models can **inherit from each other**, just like regular Python classes.  
The child model gets all fields from the parent and can add its own.

```python
class Parent(BaseModel):
    id: int
    name: str

class Child(Parent):       # inherits id and name
    extra_field: str
```

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

# Level 1 — Base with shared fields
class BaseItem(BaseModel):
    id: int
    name: str
    created_at: Optional[str] = None

# Level 2 — Adds a metadata dict
class DictItem(BaseItem):
    metadata: Dict[str, Any] = {}

# Level 3 — Adds a tags list and description
class ListItem(DictItem):
    tags: List[str] = []
    description: Optional[str] = None

# BaseItem
base = BaseItem(id=1, name="Simple Item")
print("BaseItem :", base)

# DictItem — inherits id, name, created_at
dict_item = DictItem(id=2, name="Dict Item", metadata={"source": "api", "version": 3})
print("DictItem :", dict_item)

# ListItem — inherits everything above
list_item = ListItem(
    id=3,
    name="Full Item",
    metadata={"source": "db"},
    tags=["python", "pydantic", "validation"],
    description="This is the most complete item"
)
print("ListItem :", list_item)

In [ ]:
# Checking inheritance chain
print(isinstance(list_item, ListItem))   # True
print(isinstance(list_item, DictItem))   # True  — parent
print(isinstance(list_item, BaseItem))   # True  — grandparent

print("\nAll fields on ListItem:", list(list_item.model_fields.keys()))

---
## 9. Serialization

Pydantic makes it easy to convert models back to raw data:

| Method | Returns | Use when |
|---|---|---|
| `model_dump()` | Python `dict` | Passing to other Python code |
| `model_dump_json()` | JSON `str` | Sending over HTTP / saving to file |
| `model_validate(data)` | Model instance | Parsing a dict back into a model |
| `model_validate_json(json_str)` | Model instance | Parsing a JSON string back into a model |

In [ ]:
from pydantic import BaseModel
from typing import Optional
import json

class Article(BaseModel):
    title: str
    author: str
    views: int = 0
    published: bool = True
    summary: Optional[str] = None

article = Article(title="Pydantic Guide", author="Shriman", views=1500)

# --- model_dump() → dict ---
data_dict = article.model_dump()
print("model_dump()       →", data_dict)
print("Type               →", type(data_dict))

print()

# Exclude certain fields
print("exclude published  →", article.model_dump(exclude={"published"}))

# Include only specific fields
print("include title only →", article.model_dump(include={"title"}))

In [ ]:
# --- model_dump_json() → JSON string ---
json_str = article.model_dump_json()
print("model_dump_json()  →", json_str)
print("Type               →", type(json_str))

print()

# Pretty print JSON
pretty = json.dumps(json.loads(json_str), indent=2)
print("Pretty JSON:")
print(pretty)

In [ ]:
# --- model_validate() → parse dict back into model ---
raw_data = {"title": "New Post", "author": "Alice", "views": "200"}   # views as string

parsed = Article.model_validate(raw_data)
print("Parsed model       →", parsed)
print("views type         →", type(parsed.views))   # int — auto-cast from string

# --- model_validate_json() → parse JSON string ---
json_input = '{"title": "From JSON", "author": "Bob"}'
from_json = Article.model_validate_json(json_input)
print("From JSON string   →", from_json)

---
## 10. ConfigDict — Model Configuration

`ConfigDict` controls how your model **behaves globally** — not per field but for the whole class.

```python
from pydantic import ConfigDict

class MyModel(BaseModel):
    model_config = ConfigDict(...)
```

| Option | What it does |
|---|---|
| `str_strip_whitespace=True` | Strips leading/trailing spaces from all `str` fields |
| `frozen=True` | Makes the model immutable (like a frozen dataclass) |
| `extra="forbid"` | Raises error if unknown fields are passed |
| `extra="ignore"` | Silently ignores unknown fields |
| `extra="allow"` | Accepts and stores unknown fields |
| `populate_by_name=True` | Allow using field name even when alias is set |
| `validate_default=True` | Also run validators on default values |

In [ ]:
from pydantic import BaseModel, ConfigDict

class CleanUser(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    name: str
    email: str

# Whitespace in inputs — gets stripped automatically
user = CleanUser(name="  Shriman  ", email="  shriman@example.com  ")
print(f"Name : '{user.name}'")    # 'Shriman' — stripped
print(f"Email: '{user.email}'")

In [ ]:
from pydantic import BaseModel, ConfigDict

class ImmutableConfig(BaseModel):
    model_config = ConfigDict(frozen=True)

    host: str
    port: int

cfg = ImmutableConfig(host="localhost", port=8080)
print(cfg)

# Trying to mutate a frozen model raises an error
try:
    cfg.port = 9090
except Exception as e:
    print(f"Error: {type(e).__name__} — {e}")

In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")   # No extra fields allowed

    name: str
    age: int

# Passing an unexpected field 'role'
try:
    obj = StrictModel(name="Alice", age=25, role="admin")
except ValidationError as e:
    print("Extra field error:")
    print(e)

---
## Full Example 1 — E-Commerce Order System

Combines: **nested models + Field constraints + ConfigDict + type casting + serialization**

In [ ]:
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional, Dict, Any
import json

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str = Field(min_length=5, max_length=10)
    country: str = "India"

class CartItem(BaseModel):
    product_id: str
    product_name: str
    quantity: int = Field(ge=1, le=100, description="Units to order")
    unit_price: float = Field(gt=0, description="Price per unit in INR")

    @property
    def total_price(self) -> float:
        return round(self.quantity * self.unit_price, 2)

class Order(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    order_id: str
    customer_name: str = Field(min_length=2, max_length=100)
    shipping_address: Address
    items: List[CartItem]
    coupon_code: Optional[str] = None
    notes: Optional[str] = Field(default=None, max_length=500)
    metadata: Dict[str, Any] = {}

# Create the order — strings auto-cast, whitespace stripped, dicts converted
order = Order(
    order_id="ORD-2024-001",
    customer_name="  Shriman Narayan  ",   # whitespace will be stripped
    shipping_address={
        "street": "45 MG Road",
        "city": "Bangalore",
        "state": "Karnataka",
        "zip_code": "560001"
    },
    items=[
        {"product_id": "P01", "product_name": "Keyboard", "quantity": 2, "unit_price": "1500.00"},
        {"product_id": "P02", "product_name": "Mouse",    "quantity": "1", "unit_price": "799.50"}
    ],
    metadata={"source": "web", "priority": "high"}
)

print("Customer     :", order.customer_name)              # Whitespace stripped
print("City         :", order.shipping_address.city)
print("Country      :", order.shipping_address.country)   # Default value
print("Item 1 price :", order.items[0].unit_price, type(order.items[0].unit_price))
print("Item 1 total :", order.items[0].total_price)
print("Item 2 qty   :", order.items[1].quantity, type(order.items[1].quantity))  # cast from str
print()
print("Dict dump    :", order.model_dump())
print()
print("JSON dump    :", order.model_dump_json())

---
## Full Example 2 — User Registry with Inheritance

Combines: **multi-level inheritance + Field + frozen ConfigDict + serialization**

In [ ]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, List

# Level 1 — Base user
class BaseUser(BaseModel):
    id: int
    username: str = Field(min_length=3, max_length=30)
    email: str

# Level 2 — Registered user with role and status
class RegisteredUser(BaseUser):
    is_active: bool = True
    role: str = "viewer"

# Level 3 — Admin with extra controls and immutability
class AdminUser(RegisteredUser):
    model_config = ConfigDict(frozen=True)   # Immutable after creation

    role: str = "admin"
    permissions: List[str] = Field(
        default=["read", "write"],
        description="Allowed operations"
    )
    department: Optional[str] = None

# Instances
viewer = RegisteredUser(id=1, username="john_doe", email="john@example.com")
print("Viewer:", viewer)

admin = AdminUser(
    id=2,
    username="shriman_admin",
    email="shriman@globaldata.com",
    permissions=["read", "write", "delete", "manage_users"],
    department="Data Engineering"
)
print("Admin :", admin)
print("Perms :", admin.permissions)

# Serialize
print("\nAdmin dict:", admin.model_dump())

# Parse from dict
raw = {"id": 3, "username": "new_admin", "email": "new@example.com"}
new_admin = AdminUser.model_validate(raw)
print("\nParsed admin:", new_admin)   # Gets default role='admin' and permissions

---
## Practice — Try it Yourself!

Use the cells below to practice what you've learned.

In [ ]:
# Practice 1 — Create a Book model with:
# - title (str, required)
# - author (str, required)
# - pages (int, must be >= 1)
# - genre (Optional str, default None)
# - rating (float, between 0 and 10, default 0.0)

# Your code here ↓


In [ ]:
# Practice 2 — Create a Library model that:
# - Has a name (str)
# - Holds a list of Book objects
# - Has an address (str)
# - Has total_books as Optional int, default None
# Then create a Library with 2-3 books in it.

# Your code here ↓


In [ ]:
# Practice 3 — Serialize your Library:
# - Convert it to a dict using model_dump()
# - Convert it to a JSON string using model_dump_json()
# - Parse a new Library back from a raw dict using model_validate()

# Your code here ↓


In [ ]:
# Practice 4 — Add ConfigDict to your Book model:
# - Strip whitespace from strings
# - Forbid extra fields
# Test that passing an extra field raises an error.

# Your code here ↓


---
## Quick Cheat Sheet

```python
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Dict, Any, Optional

class MyModel(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")

    name: str                                       # required
    status: str = "active"                          # default value
    nickname: Optional[str] = None                  # optional
    age: int = Field(ge=0, le=150)                  # numeric constraint
    code: str = Field(min_length=3, max_length=10)  # string length constraint
    tags: List[str] = []                            # list type
    config: Dict[str, Any] = {}                     # dict type

obj = MyModel(name="Alice", age="30", code="ABC")   # auto type-cast

obj.model_dump()                       # → dict
obj.model_dump_json()                  # → JSON string
obj.model_dump(exclude={"status"})     # → dict without 'status'
obj.model_dump(include={"name"})       # → dict with only 'name'
MyModel.model_validate({...})          # parse dict → model
MyModel.model_validate_json("{...}")   # parse JSON string → model
```